## Train the final B-Point regression model on the full feature set

This notebook fits the production model for `biopsykit.signals.icg.event_extraction.BPointExtractionAbelStuehler2025` on the **full feature set** (RR-interval + all 12 classical B-point algorithms), using the best-performing pipeline and hyperparameters identified in `evaluation/Analyse_Model_Parameters.ipynb` (`MinMax_None_RFR_RR-Interval-Include-Nan`, the previously best-performing full-feature model reported in the paper) instead of re-running the (expensive) hyperparameter search.

In [ ]:
# Imports
import pickle
import warnings
from pathlib import Path

import biopsykit as bp
import numpy as np
import pandas as pd

# Model Evaluation
from biopsykit.classification.model_selection import SklearnPipelinePermuter

# Regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model._cd_fast import ConvergenceWarning
from sklearn.metrics import mean_absolute_error

# Cross-Validation
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Datapaths
data_path = Path("../../results")
models_path = Path("../../models")
rater = "rater_01"

In [ ]:
# Load data (full feature set: RR-interval + all 12 classical B-point algorithms)
input_data_b_point = pd.read_csv(
    data_path.joinpath(f"rr-interval/{rater}/train_data_b_point_rr_interval_include_nan.csv"),
    index_col=[0, 1, 2, 3, 4, 5],
).astype(float)

In [ ]:
# Prepare data for training (split features from target and create groups)
X_b_point, y_b_point, groups_b_point, group_keys_b_point = bp.classification.utils.prepare_df_sklearn(
    data=input_data_b_point, label_col="b_point_sample_reference", subject_col="participant", print_summary=False
)
print(f"Input data dtype: {X_b_point.dtype}")
print(f"y_b_point isnan: {np.isnan(y_b_point).any()}")

# Specify model dict
model_dict_b_point = {
    "scaler": {"MinMaxScaler": MinMaxScaler()},
    "clf": {
        "RandomForestRegressor": RandomForestRegressor(),
    },
}

# Fixed hyperparameters (best-performing configuration for the full feature set, cf.
# `MinMax_None_RFR_RR-Interval-Include-Nan`
# in Analyse_Model_Parameters.ipynb / summarized_b_point_error_metrics.csv) instead of searching over a hyperparameter
# grid.
# Note: this is the same configuration that was also selected as best-performing for the reduced (`three_best_only`)
# feature set in `B_Point_Regression.ipynb`.
params_dict_b_point = {
    "MinMaxScaler": None,
    "RandomForestRegressor": {
        "bootstrap": [True],
        "criterion": ["friedman_mse"],
        "max_features": [0.8],
        "n_estimators": [250],
        "ccp_alpha": [0.001],
        "random_state": [0],
    },
}

# Grid search over the single fixed hyperparameter combination above (no actual search performed)
hyper_search_dict = {
    "RandomForestRegressor": {"search_method": "grid"},
}

# Load existing pipeline permuter or create a new one
input_file_path_b_point = models_path.joinpath(
    f"b-point/rr-interval/{rater}/b_point_include_nan_full_features_{rater}.pkl"
)

pipeline_permuter_b_point = SklearnPipelinePermuter(
    model_dict_b_point, params_dict_b_point, hyper_search_dict, random_state=0
)

# Initialize cross-validation
outer_cv = GroupKFold(n_splits=5)
inner_cv = GroupKFold(n_splits=5)
model_name = f"b_point_include_nan_full_features_{rater}.pkl"
print(f"Model name: {model_name}")

# Fit and save intermediate results
input_file_path_b_point.parent.mkdir(parents=True, exist_ok=True)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    pipeline_permuter_b_point.fit_and_save_intermediate(
        X=X_b_point,
        y=y_b_point,
        file_path=models_path.joinpath(f"b-point/rr-interval/{rater}/{model_name}"),
        outer_cv=outer_cv,
        inner_cv=inner_cv,
        scoring="neg_mean_absolute_error",
        groups=groups_b_point,
    )

# Save the fitted pipeline permuter
pipeline_permuter_b_point.to_pickle(models_path.joinpath(f"b-point/rr-interval/{rater}/{model_name}"))
print(f"Generated pickle file: {model_name}")

In [ ]:
# Simple evaluation: MAE fold-wise and sample-wise
algo_key = ("MinMaxScaler", "RandomForestRegressor")
metric_summary_b_point = pipeline_permuter_b_point.metric_summary().loc[algo_key]

true_labels_folds = metric_summary_b_point["true_labels_folds"]
predicted_labels_folds = metric_summary_b_point["predicted_labels_folds"]

# Fold-wise MAE
mae_per_fold = [mean_absolute_error(t, p) for t, p in zip(true_labels_folds, predicted_labels_folds, strict=False)]
for i, mae in enumerate(mae_per_fold):
    print(f"Fold {i}: MAE = {mae:.2f} ms")
print(f"Fold-wise MAE: {np.mean(mae_per_fold):.2f} \u00b1 {np.std(mae_per_fold):.2f} ms")

# Sample-wise MAE (all folds pooled)
abs_error_sample_wise = np.abs(
    np.asarray(metric_summary_b_point["true_labels"]) - np.asarray(metric_summary_b_point["predicted_labels"])
)
mae_sample_wise = np.mean(abs_error_sample_wise)
std_sample_wise = np.std(abs_error_sample_wise)
print(f"Sample-wise MAE: {mae_sample_wise:.2f} \u00b1 {std_sample_wise:.2f} ms")

### Fit and export the final, deployable model

The cross-validated pipelines above are only used for evaluation (each is fit on 4/5 of the data). For deployment (i.e., to be passed as `model=` to `BPointExtractionAbelStuehler2025`), fit one final pipeline on the **entire** dataset with the same fixed hyperparameters, and export it together with the feature column order it expects.

In [ ]:
# Fit the final production pipeline on all available data (same fixed hyperparameters as above)
from sklearn.pipeline import Pipeline

final_pipeline = Pipeline(
    [
        ("scaler", MinMaxScaler()),
        (
            "clf",
            RandomForestRegressor(
                bootstrap=True,
                criterion="friedman_mse",
                max_features=0.8,
                n_estimators=250,
                ccp_alpha=0.001,
                random_state=0,
            ),
        ),
    ]
)
final_pipeline.fit(X_b_point, y_b_point)

final_model_path = models_path.joinpath(
    f"b-point/rr-interval/{rater}/b_point_abelstuehler2025_full_features_{rater}.pkl"
)
# ^ this exact file name/layout matches what
# `biopsykit.signals.icg.event_extraction.get_b_point_abelstuehler2025_model()`
# expects when caching a manually-placed model under ~/.biopsykit_data/pretrained_models/, and what should be
# uploaded as the corresponding GitHub Release asset.
with final_model_path.open("wb") as f:
    pickle.dump({"model": final_pipeline, "feature_names": list(input_data_b_point.columns)}, f)
print(f"Saved final deployable model to {final_model_path}")

# Usage:
# from biopsykit.signals.icg.event_extraction import BPointExtractionAbelStuehler2025
# with open(final_model_path, "rb") as f:
#     artifact = pickle.load(f)
# algo = BPointExtractionAbelStuehler2025(model=artifact["model"])